# Sample Solutions for Packages, Imports and Project Organisation

## Exercise 1: Using the Standard Library

Either import form would do here. Importing `statistics` as a whole module and `sqrt` on its own is a reasonable mix: `statistics.mean` says plainly where `mean` came from, while `sqrt` is unambiguous enough on its own.

Note that the imports are at the top of the cell, before anything else.

In [ ]:
import statistics
from math import sqrt

readings = [4.2, 4.5, 3.9, 4.7, 4.1, 4.4, 4.6, 4.0]

print("Mean:", statistics.mean(readings))
print("Median:", statistics.median(readings))

standard_error = statistics.stdev(readings) / sqrt(len(readings))
print("Standard error:", standard_error)

The median prints as `4.300000000000001` rather than `4.3`. There are eight readings, so the median is the average of the middle two, and dividing by two leaves a tiny rounding error. Decimal numbers are stored to a limited precision, so this turns up from time to time. The value is right, the display is just showing you more of it than you wanted.

## Exercise 2: Standard Library or Installed?

Both give `4.3`. The two libraries agree, and for a job this small there is no reason to prefer one over the other.

`numpy` is the one that would have to be installed. It came from PyPI, and it is only on this machine because something put it there. `statistics` arrived with Python itself, so anyone running Python already has it.

This is also why `import` and `pip install` are different operations. `import` finds code already on the machine; `pip install` puts it there. A `ModuleNotFoundError` means the first failed, and usually that the second never happened.

In [ ]:
import statistics
import numpy as np

readings = [4.2, 4.5, 3.9, 4.7, 4.1, 4.4, 4.6, 4.0]

print("statistics.mean:", statistics.mean(readings))
print("np.mean:", np.mean(readings))

## Exercise 3: Extending Your Module

The completed `cylinders.py`:

```python
"""Functions for the geometry of cylinders."""

# This relies on circles.py being in the same directory as this file
from circles import circle_area, circle_circumference


def cylinder_volume(radius, height):
    """Return the volume of a cylinder of the given radius and height."""
    return circle_area(radius) * height


def cylinder_surface_area(radius, height):
    """Return the surface area of a cylinder of the given radius and height."""
    return 2 * circle_area(radius) + circle_circumference(radius) * height
```

Notice that the import at the top had to grow. `circle_circumference` was not being used before, so it was not imported, and adding the call without adding the import gives a `NameError`.

The new function reuses `circle_area` and `circle_circumference` rather than writing $\pi r^2$ and $2 \pi r$ out again. If either formula ever needed correcting, there would be one place to correct it, and both `cylinders.py` and anything else importing `circles.py` would be fixed at once.

If the import in the Notebook kept failing with `ImportError`, the kernel had not been restarted. Python holds on to the version of a module it imported first, and will not re-read the file.

## Exercise 4: Running the Tests

Nothing to write, but the failure in step 3 is the part to have read properly.

A failing test tells you three things: which test it was, what it expected, and what it actually got. Changing `test_no_height_means_no_volume` to expect 1 instead of 0, for instance, gives:

```
AssertionError: assert 0.0 == 1
```

against the line `assert cylinder_volume(3, 0) == 1`. That is enough to work out what is wrong without opening `cylinders.py` at all.

Note also that the other three tests still passed. A failing test tells you what is broken, and the passing ones tell you what is not, which narrows the search considerably.

## Exercise 5: Adding to the Project

The test, added to `tests/test_cylinders.py`:

```python
def test_surface_area_of_a_flat_cylinder():
    assert cylinder_surface_area(1, 0) == 2 * circle_area(1)
```

A cylinder with no height is just its two ends, so the rectangle wrapped around the side contributes nothing. That makes it an easy case to check by hand, which is what makes it a good test.

The changes to `report_cylinders.py`. The import at the top grows:

```python
from cylinders import cylinder_surface_area, cylinder_volume
```

and the loop prints both figures:

```python
for cylinder in cylinders:
    volume_cm3 = cylinder_volume(cylinder["radius_cm"], cylinder["height_cm"])
    surface_cm2 = cylinder_surface_area(cylinder["radius_cm"], cylinder["height_cm"])
    print(cylinder["label"] + ": " + str(round(volume_cm3, 1)) + " cm3, "
          + str(round(surface_cm2, 1)) + " cm2")
```

Notice that `report_cylinders.py` did not need to know how a surface area is worked out. It asked `cylinders.py` for one, which is the advantage of keeping the two jobs in separate files.

## Extra Exercise 1: Which Names Exist?

| Line | Result |
|------|-------|
| 1 | `3` - `m` is bound by `import math as m` |
| 2 | `NameError: name 'math' is not defined` |
| 3 | `4.0` - `sqrt` is bound by `from math import sqrt` |
| 4 | `4.0` - the same function, reached through the module |
| 5 | `NameError: name 'floor' is not defined` |

Line 2 is the one to look at. `import math as m` binds one name, `m`, and only that name. The alias replaces `math` rather than adding to it.

Line 5 follows the same logic from the other direction. `from math import sqrt` brings in exactly `sqrt`. Everything else in `math` is still there in the module, but no name in your code refers to it.

This exercise only behaves as described from a freshly restarted kernel. If you ran the earlier `from math import *` cell in this session, `floor` is already bound and line 5 works. That is exactly the complaint against `import *`: names arrive from somewhere off-screen, and nothing at the point of use tells you where from.

## Extra Exercise 2: The Total Volume

```python
total_volume_cm3 = 0

for cylinder in cylinders:
    volume_cm3 = cylinder_volume(cylinder["radius_cm"], cylinder["height_cm"])
    total_volume_cm3 = total_volume_cm3 + volume_cm3
    print(cylinder["label"] + ": " + str(round(volume_cm3, 1)) + " cm3")

print("")
print("Total volume: " + str(round(total_volume_cm3, 1)) + " cm3")
```

The two things to get right are both about placement. `total_volume_cm3` is set to zero *before* the loop, not inside it, or it would reset on every cylinder. And the final `print` is *outside* the loop, or you would get a running total after every line.

The answer is `10620.5` cm3. Most of that is the `mixing_vessel`, which is far larger than the rest.

## Extra Exercise 3: A Project of Your Own

No solution for this one, but a rough shape that fits most research projects:

```
my_project/
├── README.md
├── run_analysis.py
├── loading.py
├── processing.py
├── plotting.py
├── data/
└── tests/
    └── test_processing.py
```

The split between loading, processing and plotting is a common one, and it survives the one-sentence test: one module gets data in, one transforms it, one draws it. If you need the word "and" to describe a module, that is usually a sign it should be two modules.